# Barranquilla MOPAC: ¿en qué se va el tiempo de un paso QM/MM?

Tripsina–BPTI (54 átomos QM + 5 de enlace, ~46 900 átomos, PME), pasos de 0,5 fs, MOPAC en la GPU. Lo que ya se sabía
(commit a6a4992e, A100, tolerancia 0,01): un paso de ~57 ms = MOPAC ~34 ms (SCF ~27–30 ms, de ellos `diagg2` ~11,6 ms;
gradiente ~5,6; hcore ~3,1) + Python de la fuerza ~17 ms + largo alcance ~4 ms + OpenMM y traspaso ~1,5 ms. Además, en
las corridas de 400 pasos el escritor de la trayectoria PDB (~240 ms cada 40 pasos) sumaba ~6 ms por paso.

Este cuaderno mide lo mismo con el código actual, desde un único estado inicial (el de `PREP_FROM`), y prueba dos
kernels de `diagg2` que ya existen pero no se usan por defecto:

| Bloque | Qué mide | Tiempo (A100) |
|---|---|---|
| Compilar | MOPAC con OpenBLAS | ~10 min |
| A | la parte Python de la fuerza en esta CPU (MOPAC sustituido): paso completo, `compute()`, perfil por línea | ~2 min |
| B | 400 pasos NVT por variante (`nve_drift.py run --nve 0`): tolerancias 0,01 y 0,001; perfil por secciones (`MOPAC_GPU_PROFILE=1`); traza de `diagg2` (`MOPAC_MOZYME_SCF_TRACE=1`); `diagg2` *staged* y cooperativo; repeticiones; OpenMM solo | ~12 min |
| B2 | el script de producción (`run_qmmm_md.py`), para medir el escritor de la trayectoria | ~2 min |
| C | `nsys` de 40 llamadas, si está instalado: kernels, llamadas a la API de CUDA, tiempo ocupado de la GPU | ~3 min |

Todas las corridas de B parten del mismo estado con la misma semilla, así que la primera llamada a MOPAC es la misma
geometría en todas: su calor de formación sirve para comprobar que los kernels alternativos dan el mismo resultado.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a
ejecutar todo y lo terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '89a30daf'            # nve_drift.py --nve 0
RESULTS_NAME = 'qmmm_prof_a100'
PREP_FROM = ['qmmm_dt_a100', 'qmmm_guess2_a100']   # estado inicial (el primero que exista); si no, uno nuevo
NVT_STEPS = 400                # pasos de 0,5 fs por variante
USE_DRIVE = True
QUICK = False                  # True: 40 pasos (prueba del cuaderno)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
else:
    BASE = Path('/content/mopac_paper_bench')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM, CuPy y line_profiler

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} line_profiler 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available(), '; fused kernels:', fused_far_field_available())\"",
    check=False)

## 5. Herramientas y estado inicial

- `/content/qmmm_prof.py` ejecuta un script QM/MM sin cambiarlo y guarda, por llamada a la fuerza QM/MM: inicio,
  duración de `compute()`, de MOPAC y del largo alcance, calor de formación, iteraciones SCF y cargas MM vistas.
- `/content/prof_py.py`: la parte Python sola (MOPAC sustituido por una función instantánea).
- `/content/analyze_prof.py`: lectura de los perfiles y de `nsys`.

In [ ]:
Path('/content/qmmm_prof.py').write_text('"""Runs a QM/MM script unchanged (python qmmm_prof.py SCRIPT ARGS...) and records each call of the QM/MM force:\nstart time, duration of compute(), of MOPAC (QMSession.step) and of the long-range term, heat (kcal/mol),\nSCF iterations and MM charges seen.  QMMM_PROF_OUT: JSON file of these lists.  QMMM_NSYS_RANGE=a,b: CUDA\nprofiler start before call a and stop after call b (nsys --capture-range=cudaProfilerApi)."""\nimport json\nimport os\nimport runpy\nimport sys\nimport time\n\nscript = sys.argv[1]\nsys.argv = sys.argv[1:]\nsys.path.insert(0, os.path.dirname(os.path.abspath(script)))\nimport openmm_mopac as om  # noqa: E402\nimport mopac_qmmm_api as api  # noqa: E402\n\nT = {k: [] for k in ("t0", "compute", "mopac", "pme_far", "pme_grad", "heat", "iters", "n_mm")}\nrng = os.environ.get("QMMM_NSYS_RANGE")\nrng = tuple(int(x) for x in rng.split(",")) if rng else None\nprof = None\nif rng:\n    try:\n        import cupy\n        prof = cupy.cuda.runtime\n    except Exception as e:  # noqa: BLE001\n        print("QMMM_NSYS_RANGE: no CuPy,", e, flush=True)\n\n\ndef wrap(cls, name, key):\n    f = getattr(cls, name)\n\n    def g(self, *a, **k):\n        t = time.perf_counter()\n        try:\n            return f(self, *a, **k)\n        finally:\n            T[key].append(time.perf_counter() - t)\n    setattr(cls, name, g)\n\n\nwrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")\nwrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")\nwrap(api.QMSession, "step", "mopac")\n_compute = om.MopacQMMMForce.compute\n\n\ndef compute(self, state):\n    k = len(T["t0"])\n    if prof is not None and k == rng[0]:\n        prof.deviceSynchronize()\n        prof.profilerStart()\n    t = time.perf_counter()\n    T["t0"].append(t)\n    r = _compute(self, state)\n    T["compute"].append(time.perf_counter() - t)\n    T["heat"].append(self.last.get("heat"))\n    T["iters"].append(self.session.scf_iterations)\n    T["n_mm"].append(self.last.get("n_mm"))\n    if prof is not None and k == rng[1]:\n        prof.deviceSynchronize()\n        prof.profilerStop()\n    return r\n\n\nom.MopacQMMMForce.compute = compute\ntry:\n    runpy.run_path(script, run_name="__main__")\nfinally:\n    out = os.environ.get("QMMM_PROF_OUT")\n    if out:\n        with open(out, "w") as fh:\n            json.dump(T, fh)\n')
Path('/content/prof_py.py').write_text('"""The Python side of the QM/MM force on this machine, MOPAC replaced by an instant deterministic stub.\n    python prof_py.py SCRIPTS_DIR PREP_DIR OUT_JSON LPROF_TXT [PLATFORM]\nTimes (1) whole Verlet steps with the QM/MM PythonForce (OpenMM + hand-over of positions and forces + compute),\n(2) compute() and its long-range part per call, and (3) writes a line profile of compute() and of the\nlong-range class in use."""\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport openmm as mm\nimport openmm.app as app\nimport openmm.unit as u\n\nsys.path.insert(0, sys.argv[1])\nimport openmm_mopac as om  # noqa: E402\nimport qmmm_pme as qp  # noqa: E402\nfrom nve_drift import FF, make_system  # noqa: E402\nfrom run_qmmm_md import BACKBONE, parse_residues, residue_matches  # noqa: E402\n\nprep, out_json, out_txt = sys.argv[2], sys.argv[3], sys.argv[4]\nplat = sys.argv[5] if len(sys.argv) > 5 else "CUDA"\n\n\nclass Stub:\n    def __init__(self, libpath, nums, x0, charge=0, model=0, tolerance=0.01, guess=None):\n        self.n = len(nums)\n        self.scf_iterations = 0\n\n    def step(self, xq, xs, qs, ron, roff, qm_pot=None):\n        xs = np.reshape(xs, (-1, 3))\n        d = xs - np.mean(xq, axis=0)\n        self.mm_pot = 0.1 * np.sin(7.0 * qs + 0.3 * d[:, 0])\n        self.qm_charges = 0.05 * np.cos(np.arange(self.n))\n        return float(np.sum(qs * self.mm_pot)), 0.02 * (xq - xq.mean(axis=0)), 0.01 * d\n\n    def close(self):\n        pass\n\n\nom.QMSession = Stub\nT = {"compute": [], "far": [], "grad": []}\norig_compute = om.MopacQMMMForce.compute\norig_far, orig_grad = om.MopacQMMMForce._pme_far_potential, om.MopacQMMMForce._pme_far_gradient\n\n\ndef timed(f, key):\n    def g(*a):\n        t = time.perf_counter()\n        r = f(*a)\n        T[key].append(time.perf_counter() - t)\n        return r\n    return g\n\n\nom.MopacQMMMForce.compute = timed(orig_compute, "compute")\nom.MopacQMMMForce._pme_far_potential = timed(orig_far, "far")\nom.MopacQMMMForce._pme_far_gradient = timed(orig_grad, "grad")\n\nff = app.ForceField(*FF)\ntop = app.PDBFile(os.path.join(prep, "top.pdb")).topology\nwith open(os.path.join(prep, "state.xml")) as fh:\n    st0 = mm.XmlSerializer.deserialize(fh.read())\nsystem = make_system(ff, top, False)\nqm = [x.index for x in top.atoms() for spec, sc in parse_residues("E:57:sc,E:102:sc,E:195:sc,I:15,I:16")\n      if residue_matches(x.residue, spec) and not (sc and x.name in BACKBONE)]\nqmmm = om.add_qmmm(system, top, st0.getPositions(), qm, 0, "stub", ron=10.0, roff=12.0)\np = mm.Platform.getPlatformByName(plat)\nprops = {"Precision": "mixed"} if plat in ("CUDA", "OpenCL") else {}\ninteg = mm.VerletIntegrator(0.5 * u.femtoseconds)\nctx = mm.Context(system, integ, p, props)\nctx.setPeriodicBoxVectors(*st0.getPeriodicBoxVectors())\nctx.setPositions(st0.getPositions())\nctx.setVelocities(st0.getVelocities())\ninteg.step(5)                                      # grid and kernels set up\nres = {"platform": plat, "atoms": system.getNumParticles(), "qm_atoms": len(qm), "pme_gpu": qmmm.region.pme_gpu,\n       "far_class": type(qmmm.far).__name__ if qmmm.far is not None else None}\nn = 200\nfor k in T:\n    T[k].clear()\nt0 = time.perf_counter()\ninteg.step(n)\nres["step_ms"] = (time.perf_counter() - t0) / n * 1e3\nres["compute_ms"] = float(np.median(T["compute"]) * 1e3)\nres["far_ms"] = float(np.median(np.array(T["far"]) + np.array(T["grad"])) * 1e3) if T["far"] else 0.0\nres["python_rest_ms"] = res["compute_ms"] - res["far_ms"]\nres["outside_compute_ms"] = res["step_ms"] - float(np.mean(T["compute"]) * 1e3)\nres["n_mm"] = qmmm.last["n_mm"]\nprint(json.dumps(res, indent=1), flush=True)\nwith open(out_json, "w") as fh:\n    json.dump(res, fh, indent=1)\n\ntry:\n    from line_profiler import LineProfiler\nexcept ImportError:\n    print("line_profiler not installed: no line profile", flush=True)\n    sys.exit(0)\nstate = ctx.getState(getPositions=True)\nlp = LineProfiler()\nom.MopacQMMMForce._pme_far_potential, om.MopacQMMMForce._pme_far_gradient = orig_far, orig_grad\nfor cls in (qp.FarFieldGPU, qp.FarFieldNumPy):\n    cls.potential = lp(cls.potential)\n    cls.gradient = lp(cls.gradient)\nprof = lp(orig_compute)\nfor _ in range(100):\n    prof(qmmm, state)\nwith open(out_txt, "w") as fh:\n    lp.print_stats(stream=fh, output_unit=1e-3, stripzeros=True)\nprint("line profile:", out_txt, flush=True)\n')
Path('/content/analyze_prof.py').write_text('import re\nimport sqlite3\nfrom collections import defaultdict\n\nimport numpy as np\n\nRX_SEC = re.compile(r\'\\[PROFILE\\] MOZYME_SECTION name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_STAGE = re.compile(r\'\\[PROFILE\\] MOZYME_RESIDENT_STAGE name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_TRACE = re.compile(r\'\\[MOZYME GPU SCF trace\\] diagg2 staged=(\\d+) nij=(\\d+) rotated=(\\d+) max_chain=(\\d+) \'\n                      r\'lock_spins=(\\d+) nrej=(-?\\d+) lpt=(\\d+) grid=(\\d+) span_us=([0-9.]+) wait_us=([0-9.]+) \'\n                      r\'rotate_us=([0-9.]+)\')\n\n\ndef steady(T, skip=3):\n    """per-call medians (ms) after the first `skip` calls (the first SCF starts from the Lewis structure)"""\n    c, t0, m = np.array(T[\'compute\']), np.array(T[\'t0\']), np.array(T[\'mopac\'])\n    n = len(c)\n    far = (np.array(T[\'pme_far\']) + np.array(T[\'pme_grad\'])) if len(T[\'pme_far\']) == n else np.zeros(n)\n    s = slice(skip, n)\n    gap = t0[skip + 1:] - (t0[skip:-1] + c[skip:-1])\n    it = [x for x in T[\'iters\'][skip:] if x is not None]\n    r = dict(calls=n, step_ms=(t0[-1] + c[-1] - t0[skip]) / (n - skip) * 1e3,\n             compute_ms=np.median(c[s]) * 1e3, mopac_ms=np.median(m[s]) * 1e3, far_ms=np.median(far[s]) * 1e3,\n             python_ms=np.median(c[s] - m[s] - far[s]) * 1e3, gap_ms=np.median(gap) * 1e3,\n             gap_mean_ms=np.mean(gap) * 1e3, gaps_over_50ms=int(np.sum(gap > 0.05)),\n             scf_iterations=float(np.mean(it)) if it else None, mopac_ms_per_iteration=None,\n             first_call_ms=c[0] * 1e3, first_heat=T[\'heat\'][0], first_iters=T[\'iters\'][0])\n    if it:\n        r[\'mopac_ms_per_iteration\'] = r[\'mopac_ms\'] / np.mean(it)\n    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}\n\n\ndef sections(text):\n    """MOPAC\'s cumulative section times (MOPAC_API_OUTPUT): the last value of each section"""\n    sec = {}\n    for m in RX_SEC.finditer(text):\n        sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))\n    return sec\n\n\ndef stages(text):\n    """resident SCF stages (stdout), summed over the SCF calls"""\n    tot = defaultdict(lambda: [0, 0.0])\n    for m in RX_STAGE.finditer(text):\n        tot[m.group(1)][0] += int(m.group(2))\n        tot[m.group(1)][1] += float(m.group(3))\n    return dict(tot)\n\n\ndef trace(text):\n    """diagg2 sweeps (MOPAC_MOZYME_SCF_TRACE=1): medians over the sweeps"""\n    rows = [tuple(float(x) for x in m.groups()) for m in RX_TRACE.finditer(text)]\n    if not rows:\n        return {}\n    a = np.array(rows)\n    names = [\'staged\', \'nij\', \'rotated\', \'max_chain\', \'lock_spins\', \'nrej\', \'lpt\', \'grid\', \'span_us\', \'wait_us\',\n             \'rotate_us\']\n    r = {\'sweeps\': len(rows)}\n    r.update({n: float(np.median(a[:, i])) for i, n in enumerate(names)})\n    r[\'us_per_rotation_on_critical_path\'] = float(np.median(a[:, 8] / np.maximum(a[:, 3], 1)))\n    return r\n\n\ndef _union(iv):\n    iv = sorted(iv)\n    tot, cs, ce = 0, None, None\n    for s, e in iv:\n        if cs is None or s > ce:\n            if cs is not None:\n                tot += ce - cs\n            cs, ce = s, e\n        else:\n            ce = max(ce, e)\n    if cs is not None:\n        tot += ce - cs\n    return tot\n\n\ndef nsys_summary(sqlite_path, ncalls):\n    """GPU busy time against wall time over the captured range, launches and API calls per QM/MM call"""\n    con = sqlite3.connect(sqlite_path)\n    tables = {r[0] for r in con.execute("select name from sqlite_master where type=\'table\'")}\n    names = dict(con.execute(\'select id, value from StringIds\'))\n    k = con.execute(\'select start, end, shortName from CUPTI_ACTIVITY_KIND_KERNEL\').fetchall()\n    mc = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMCPY\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMCPY\' in tables else []\n    ms = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMSET\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMSET\' in tables else []\n    rt = con.execute(\'select start, end, nameId from CUPTI_ACTIVITY_KIND_RUNTIME\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_RUNTIME\' in tables else []\n    t_lo = min([r[0] for r in rt] + [r[0] for r in k])\n    t_hi = max([r[1] for r in rt] + [r[1] for r in k])\n    wall = (t_hi - t_lo) * 1e-6\n    busy = _union([(s, e) for s, e, _ in k] + mc + ms) * 1e-6\n    kern = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in k:\n        kern[names.get(nid, str(nid))][0] += 1\n        kern[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    api = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in rt:\n        api[names.get(nid, str(nid))][0] += 1\n        api[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    n = max(1, ncalls)\n    return dict(calls=ncalls, wall_ms_per_call=wall / n, gpu_busy_ms_per_call=busy / n,\n                kernel_ms_per_call=sum(v[1] for v in kern.values()) / n,\n                kernels_per_call=len(k) / n, memcpy_per_call=len(mc) / n, memset_per_call=len(ms) / n,\n                api_calls_per_call=len(rt) / n,\n                top_kernels=[(name, c / n, t / n) for name, (c, t) in sorted(kern.items(), key=lambda x: -x[1][1])[:30]],\n                top_api=[(name, c / n, t / n) for name, (c, t) in sorted(api.items(), key=lambda x: -x[1][1])[:20]])\n')

sys.path.insert(0, '/content')
import analyze_prof as AP
import urllib.request
PROF = Path('/content/qmmm_prof.py')
QM = SRC / 'scripts' / 'qmmm'
ND = QM / 'nve_drift.py'
prep = OUT / 'runs' / 'prep'
if not (prep / 'state.xml').exists():
    prep.mkdir(parents=True, exist_ok=True)
    src = next((BASE / p / 'runs' / 'C_drift' / 'prep' for p in PREP_FROM
                if (BASE / p / 'runs' / 'C_drift' / 'prep' / 'state.xml').exists()), None)
    if src:
        for f in ('state.xml', 'top.pdb'):
            shutil.copy2(src / f, prep / f)
        (prep / 'from.txt').write_text(str(src) + '\n')
        print('estado inicial copiado de', src)
    else:
        pdb = OUT / 'runs' / '_pdb' / '2PTC.pdb'
        pdb.parent.mkdir(parents=True, exist_ok=True)
        if not pdb.exists():
            urllib.request.urlretrieve('https://files.rcsb.org/download/2PTC.pdb', pdb)
        print('no hay estado de', PREP_FROM, '; se prepara uno nuevo (50 ps de MM)')
        run(f'{sys.executable} -u {ND} prepare {pdb} --out {prep} --padding 1.0 --variants E:57=HID '
            f'--mm-equil-ps {10 if QUICK else 50} --platform CUDA --precision mixed', log=prep / 'log.txt')
N = 40 if QUICK else NVT_STEPS
QMSEL = '--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --ron 10 --roff 12 --precision mixed'
print('prep:', (prep / 'from.txt').read_text().strip() if (prep / 'from.txt').exists() else prep)

## A. La parte Python de la fuerza, en esta CPU

MOPAC sustituido por una función instantánea; el largo alcance en la GPU como en producción. `step_ms`: un paso
Verlet completo (OpenMM + traspaso de posiciones y fuerzas + `compute()`); `outside_compute_ms`: lo que no es
`compute()`. El perfil por línea va a `line_profile.txt`.

In [ ]:
dA = OUT / 'runs' / 'A_python'
dA.mkdir(parents=True, exist_ok=True)
if not (dA / 'python_part.json').exists():
    run(f'{sys.executable} -u /content/prof_py.py {QM} {prep} {dA}/python_part.json {dA}/line_profile.txt CUDA',
        check=False, log=dA / 'log.txt')
print((dA / 'python_part.json').read_text() if (dA / 'python_part.json').exists() else 'sin resultado (ver log.txt)')

## B. Variantes, 400 pasos NVT de 0,5 fs cada una (`nve_drift.py run --nve 0`)

| Corrida | Variables de entorno |
|---|---|
| `tol0.01`, `tol0.001` | ninguna (tiempos de referencia) |
| `*_prof` | `MOPAC_GPU_PROFILE=1`: tiempo por sección de MOPAC y por etapa del SCF residente (añade sincronizaciones: sus tiempos totales no sirven de referencia) |
| `tol0.01_trace` | `MOPAC_MOZYME_SCF_TRACE=1`, 100 pasos: por barrido de `diagg2`, rotaciones, cadena más larga, esperas |
| `*_staged` | `MOPAC_MOZYME_DIAGG2_STAGED=1`: `diagg2` con los coeficientes en memoria compartida |
| `*_coop` | `MOPAC_MOZYME_DIAGG2_COOPERATIVE=1`: el `diagg2` cooperativo anterior |
| `*_rep2` | repetición, para ver cuánto varían los tiempos entre corridas iguales |
| `mm_only` | sin región QM: lo que tarda OpenMM solo |

Una corrida que falla queda registrada y las demás siguen.

In [ ]:
STAGED = {'MOPAC_MOZYME_DIAGG2_STAGED': '1'}
COOP = {'MOPAC_MOZYME_DIAGG2_COOPERATIVE': '1'}
PROFILE = {'MOPAC_GPU_PROFILE': '1'}
VARIANTS = [('tol0.01', 0.01, {}, N), ('tol0.01_staged', 0.01, STAGED, N), ('tol0.01_coop', 0.01, COOP, N),
            ('tol0.001', 0.001, {}, N), ('tol0.01_prof', 0.01, PROFILE, N), ('tol0.001_prof', 0.001, PROFILE, N),
            ('tol0.01_staged_prof', 0.01, {**STAGED, **PROFILE}, N), ('tol0.01_coop_prof', 0.01, {**COOP, **PROFILE}, N),
            ('tol0.01_trace', 0.01, {'MOPAC_MOZYME_SCF_TRACE': '1'}, min(N, 100)),
            ('tol0.01_rep2', 0.01, {}, N), ('tol0.01_staged_rep2', 0.01, STAGED, N), ('tol0.01_coop_rep2', 0.01, COOP, N),
            ('mm_only', None, {}, N)]

def save_profiles(d, api_out, log_text, T):
    if T and T.get('compute'):
        (d / 'result.json').write_text(json.dumps(AP.steady(T), indent=1))
    sec = AP.sections(api_out.read_text(errors='ignore')) if api_out.exists() else {}
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    st = AP.stages(log_text)
    if st:
        with open(d / 'stages.csv', 'w') as fh:
            fh.write('stage,calls,ms_total\n')
            for k, (c, ms) in sorted(st.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    tr = AP.trace(log_text)
    if tr:
        (d / 'trace.json').write_text(json.dumps(tr, indent=1))
    if api_out.exists():
        if sec:
            with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
                shutil.copyfileobj(fi, fo)
        api_out.unlink()

failed = []
for name, tol, envx, nvt in VARIANTS:
    d = OUT / 'runs' / 'B' / name
    opts = '--mm-only --precision mixed' if tol is None else f'{QMSEL} --tolerance {tol}'
    cmd = (f'{sys.executable} -u {PROF} {ND} run --prep {prep} --out {d} --libmopac {LIBMOPAC} {opts} '
           f'--platform CUDA --dt 0.5 --nvt {nvt} --nve 0 --seed 11')
    key = cmd + '\n' + json.dumps(envx, sort_keys=True)
    if (d / 'summary.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        print(name, 'ya hecho'); continue
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{name}.out'
    api_out.unlink(missing_ok=True)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'), **envx)
    t0 = time.time()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    T = json.loads((d / 'pytimes.json').read_text()) if (d / 'pytimes.json').exists() else None
    save_profiles(d, api_out, text, T)
    r = json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
    s = json.loads((d / 'summary.json').read_text()) if (d / 'summary.json').exists() else {}
    print(f"{name}: rc {rc}, {time.time() - t0:.0f} s; NVT {s.get('nvt_s_per_step', float('nan')) * 1e3:.1f} ms/step; "
          f"per call (median, ms): step {r.get('step_ms')}, MOPAC {r.get('mopac_ms')}, far {r.get('far_ms')}, "
          f"Python {r.get('python_ms')}, gap {r.get('gap_ms')}; SCF iterations {r.get('scf_iterations')}; "
          f"first heat {r.get('first_heat')}", flush=True)
    if rc != 0:
        failed.append(name)
        print('    FALLÓ; final del registro:\n' + '\n'.join(text.strip().splitlines()[-15:]))
print('fallidas:', failed or 'ninguna')

## B2. El script de producción (`run_qmmm_md.py`)

Tolerancia 0,01, 400 pasos, con su escritor de trayectoria (`qmmm_traj.pdb`, un fotograma cada 40 pasos). Prepara
su propio sistema (otra solvatación). Los huecos de más de 50 ms entre llamadas son el escritor.

In [ ]:
dB2 = OUT / 'runs' / 'B2_production'
pdb = OUT / 'runs' / '_pdb' / '2PTC.pdb'
pdb.parent.mkdir(parents=True, exist_ok=True)
if not pdb.exists():
    urllib.request.urlretrieve('https://files.rcsb.org/download/2PTC.pdb', pdb)
if not (dB2 / 'result.json').exists():
    dB2.mkdir(parents=True, exist_ok=True)
    cmd = (f'{sys.executable} -u {PROF} {QM}/run_qmmm_md.py {pdb} --libmopac {LIBMOPAC} '
           f'--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
           f'--tolerance 0.01 --platform CUDA --steps {N} --nve 0 --mm-equil-ps 10 --seed 11')
    (dB2 / 'cmd.txt').write_text(cmd)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1',
               QMMM_PROF_OUT=str(dB2 / 'pytimes.json'))
    with open(dB2 / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=dB2, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    if (dB2 / 'pytimes.json').exists():
        import numpy as np
        T = json.loads((dB2 / 'pytimes.json').read_text())
        r = AP.steady(T)
        c, t0 = np.array(T['compute']), np.array(T['t0'])
        gap = t0[1:] - (t0[:-1] + c[:-1])
        r['writer_ms_per_step'] = float(gap[gap > 0.05].sum() / max(1, len(c) - 3) * 1e3)
        m = re.search(r'QM/MM NVT: .* = ([0-9.]+) s/step', (dB2 / 'log.txt').read_text(errors='ignore'))
        r['reported_s_per_step'] = float(m.group(1)) if m else None
        (dB2 / 'result.json').write_text(json.dumps(r, indent=1))
    for f in dB2.glob('qmmm_traj.pdb'):
        f.unlink()
print((dB2 / 'result.json').read_text() if (dB2 / 'result.json').exists() else 'sin resultado (ver log.txt)')

## C. `nsys` (si está instalado)

Llamadas 20 a 59 (40 pasos) de la variante `tol0.01`, con la captura limitada por `cudaProfilerStart/Stop`. Se
guardan los resúmenes de `nsys stats` y, del fichero sqlite, el tiempo ocupado de la GPU por paso frente al tiempo
de reloj, y los kernels y llamadas a la API de CUDA por paso. El informe `.nsys-rep` queda en Drive (no en el zip).

In [ ]:
import glob
NSYS = shutil.which('nsys') or next(iter(sorted(glob.glob('/usr/local/cuda*/bin/nsys') +
                                                glob.glob('/opt/nvidia/nsight-systems*/**/bin/nsys', recursive=True))), None)
print('nsys:', NSYS)
dC = OUT / 'runs' / 'C_nsys'
if NSYS and not (dC / 'nsys_summary.json').exists():
    for attempt, rng in (('range', '20,59'), ('whole', None)):
        shutil.rmtree(dC, ignore_errors=True); dC.mkdir(parents=True)
        nvt = 80 if rng else 40
        env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1',
                   QMMM_PROF_OUT=str(dC / 'pytimes.json'))
        cap = ''
        if rng:
            env['QMMM_NSYS_RANGE'] = rng
            cap = '--capture-range=cudaProfilerApi --capture-range-end=stop '
        cmd = (f'{NSYS} profile -t cuda,nvtx -s none --cpuctxsw=none {cap}-o {dC}/rep -f true '
               f'{sys.executable} -u {PROF} {ND} run --prep {prep} --out {dC}/run --libmopac {LIBMOPAC} {QMSEL} '
               f'--tolerance 0.01 --platform CUDA --dt 0.5 --nvt {nvt} --nve 0 --seed 11')
        (dC / 'cmd.txt').write_text(cmd)
        with open(dC / 'log.txt', 'w') as fh:
            rc = subprocess.run(cmd, shell=True, cwd=dC, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
        rep = next(dC.glob('rep*.nsys-rep'), None)
        print(attempt, 'rc', rc, 'report', rep)
        if not rep:
            continue
        subprocess.run(f'{NSYS} stats -r cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum -f csv -o {dC}/stats {rep}',
                       shell=True, stdout=open(dC / 'stats_log.txt', 'w'), stderr=subprocess.STDOUT)
        sq = next(dC.glob('rep*.sqlite'), None)
        try:
            ncalls = 40 if rng else len(json.loads((dC / 'pytimes.json').read_text())['compute'])
            s = AP.nsys_summary(str(sq), ncalls)
            s['capture'] = attempt
            if s['kernels_per_call'] > 0:
                (dC / 'nsys_summary.json').write_text(json.dumps(s, indent=1))
        except Exception as e:
            print('sqlite:', repr(e))
        if sq:
            sq.unlink()
        if (dC / 'nsys_summary.json').exists():
            break
if (dC / 'nsys_summary.json').exists():
    s = json.loads((dC / 'nsys_summary.json').read_text())
    print({k: v for k, v in s.items() if not k.startswith('top_')})
    for name, c, t in s['top_kernels'][:15]:
        print(f'  {t:8.3f} ms/paso  {c:7.1f} veces/paso  {name[:90]}')
    for name, c, t in s['top_api'][:10]:
        print(f'  {t:8.3f} ms/paso  {c:7.1f} veces/paso  {name[:90]}')

## D. Resultados

- `prof_runs.csv`: por variante, mediana por llamada (ms) de: paso completo (`step_ms`, de reloj), `compute()`,
  MOPAC, largo alcance, Python (`compute()` menos MOPAC y largo alcance) y hueco entre llamadas (OpenMM y traspaso);
  iteraciones SCF por paso; calor de la primera llamada (misma geometría en todas).
- `prof_sections.csv` / `prof_stages.csv`: ms por paso de cada sección de MOPAC y de cada etapa del SCF residente.

In [ ]:
import pandas as pd
import numpy as np
rows, sec_rows, st_rows = [], [], []
for d in sorted((OUT / 'runs' / 'B').iterdir()):
    r = json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
    s = json.loads((d / 'summary.json').read_text()) if (d / 'summary.json').exists() else {}
    r.update(name=d.name, nvt_ms_per_step=s.get('nvt_s_per_step', np.nan) * 1e3, ok=(d / 'summary.json').exists())
    rows.append(r)
    calls = r.get('calls') or 1
    if (d / 'sections.csv').exists():
        for _, x in pd.read_csv(d / 'sections.csv').iterrows():
            sec_rows.append(dict(name=d.name, section=x.section, ms_per_step=x.ms_total / calls, calls_per_step=x.calls / calls))
    if (d / 'stages.csv').exists():
        for _, x in pd.read_csv(d / 'stages.csv').iterrows():
            st_rows.append(dict(name=d.name, stage=x.stage, ms_per_step=x.ms_total / calls, calls_per_step=x.calls / calls))
runs_df = pd.DataFrame(rows)
base = runs_df.set_index('name')
if 'tol0.01' in base.index and 'first_heat' in base:
    h0 = base.loc['tol0.01', 'first_heat']
    runs_df['first_heat_minus_tol0.01'] = runs_df['first_heat'] - h0
runs_df.to_csv(OUT / 'prof_runs.csv', index=False)
pd.DataFrame(sec_rows).to_csv(OUT / 'prof_sections.csv', index=False)
pd.DataFrame(st_rows).to_csv(OUT / 'prof_stages.csv', index=False)
pd.set_option('display.width', 250)
display(runs_df[[c for c in ['name', 'ok', 'nvt_ms_per_step', 'step_ms', 'compute_ms', 'mopac_ms', 'far_ms', 'python_ms',
                             'gap_ms', 'scf_iterations', 'mopac_ms_per_iteration', 'first_iters', 'first_heat',
                             'first_heat_minus_tol0.01'] if c in runs_df]])
if sec_rows:
    sd = pd.DataFrame(sec_rows).pivot_table(index='section', columns='name', values='ms_per_step')
    display(sd.sort_values(sd.columns[0], ascending=False).head(25).round(3))
if st_rows:
    sd = pd.DataFrame(st_rows).pivot_table(index='stage', columns='name', values='ms_per_step')
    display(sd.sort_values(sd.columns[0], ascending=False).round(3))
tr = OUT / 'runs' / 'B' / 'tol0.01_trace' / 'trace.json'
print('diagg2 (traza):', tr.read_text() if tr.exists() else 'sin traza')
for f in (OUT / 'runs' / 'A_python' / 'python_part.json', OUT / 'runs' / 'B2_production' / 'result.json'):
    print(f.parent.name, f.read_text() if f.exists() else 'sin resultado')

## E. Descargar

In [ ]:
tmp = Path('/content/_qmmm_prof') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('prof_runs.csv', 'prof_sections.csv', 'prof_stages.csv', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', '*.csv', '*.txt', 'mopac_api.out.gz'):
    for f in (OUT / 'runs').rglob(pat):
        if '_pdb' in f.parts:
            continue
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)